# Modelo genérico · K-Means implementado do zero

**Template reutilizável** de clustering: atribuição ao centroide mais próximo, recálculo dos centroides, inicialização aleatória ou k-means++, várias inicializações, escolha de *k* (cotovelo e silhouette), perfil dos grupos e comparação com o `KMeans` do scikit-learn. Inclui, opcionalmente, a aplicação do original: **compressão de imagem** por quantização de cores.

| | |
|---|---|
| **Modelo no sistema** | Machine Learning › 3. Aprendizado não supervisionado › cartão *K-Means* (`/modelar/aprendizado-nao-supervisionado/#m-kmeans`) |
| **Origem** | `C3_W1_KMeans_Assignment.ipynb` (*Machine Learning Specialization*, DeepLearning.AI e Stanford Online; conjunto 2-D de exemplo; compressão de uma foto para 16 cores), revisado, corrigido e generalizado |
| **Tipo de problema** | Não supervisionado · clustering |
| **Dependências** | `numpy`, `pandas`, `matplotlib`, `scikit-learn` |

### Como usar
1. Edite apenas a célula **1. Configuração** (`>>> ADAPTAR`).
2. Com `CAMINHO_DADOS = None`, roda com grupos sintéticos.
3. Para a compressão de imagem, informe `CAMINHO_IMAGEM` (PNG ou JPG); com `None`, a seção é pulada.

### O que mudou em relação ao notebook original
- **Padronização** das features (o cartão a considera obrigatória: sem ela, a feature de maior escala domina a distância). O original rodava nos valores crus.
- `compute_centroids` produzia **`nan`** quando um grupo ficava vazio (média de zero pontos); agora o centroide vazio é reposicionado no ponto mais distante do seu centroide atual.
- O original rodava 10 iterações fixas e **uma** inicialização. Agora há critério de convergência, `N_INICIALIZACOES` (fica a de menor distorção, como o próprio original recomendava no texto) e inicialização k-means++.
- Entrou a **escolha de k** (cotovelo da inércia e silhouette), que o original não tinha, e o perfil dos grupos nas unidades originais.
- `find_closest_centroids` virou vetorizada (a versão em laço do original continua como referência e é conferida); `run_kMeans` não cria mais uma figura vazia quando não desenha nada.
- `utils.py` e `public_tests.py` (ausentes) foram removidos; os gráficos de progresso e de cores RGB foram reescritos e generalizados. A conversão de JPG (`/255`) passou a ser automática.

## 1. Configuração

In [ ]:
# >>> ADAPTAR: única célula que precisa ser editada para usar outro dataset.

# --- dados ---
CAMINHO_DADOS = None        # ex.: "dados/clientes.csv"; None = dados sintéticos
SEPARADOR = None
TEM_CABECALHO = True        # False se a primeira linha já é dado
NOMES_COLUNAS = None        # nomes quando TEM_CABECALHO = False; sem nomes, viram coluna_0, coluna_1, …
COLUNAS_FEATURES = None     # None = todas as colunas numéricas
PADRONIZAR = True           # recomendado pelo cartão

# --- K-Means ---
K = 4                       # número de grupos do modelo final (revise após a seção 5)
FAIXA_K = range(2, 11)      # valores de k avaliados no cotovelo e na silhouette
INICIALIZACAO = "k-means++" # "k-means++" ou "aleatoria" (como no original)
N_INICIALIZACOES = 10
MAX_ITERS = 100             # o original usava 10
TOLERANCIA = 1e-6           # para quando os centroides quase não se movem
SEMENTE = 42

# --- compressão de imagem (opcional) ---
CAMINHO_IMAGEM = None       # ex.: "imagens/foto.png"; None = pula a seção
K_IMAGEM = 16               # número de cores da imagem comprimida

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, davies_bouldin_score, silhouette_score
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(SEMENTE)

## 2. Carregamento e preparação

In [ ]:
def carregar_tabela(caminho, separador=None, tem_cabecalho=True, nomes_colunas=None, planilha=0):
    """Lê CSV/TXT/TSV, Excel, Parquet ou JSON e devolve um DataFrame."""
    caminho = Path(caminho)
    if not caminho.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho.resolve()}")
    ext = caminho.suffix.lower()
    cabecalho = 0 if tem_cabecalho else None
    if ext in {".csv", ".txt", ".tsv"}:
        sep = separador if separador is not None else ("\t" if ext == ".tsv" else None)
        df = pd.read_csv(caminho, sep=sep, engine="python", header=cabecalho)
    elif ext in {".xlsx", ".xls"}:
        df = pd.read_excel(caminho, sheet_name=planilha, header=cabecalho)
    elif ext == ".parquet":
        df = pd.read_parquet(caminho)
    elif ext == ".json":
        df = pd.read_json(caminho)
    else:
        raise ValueError(f"Formato não suportado: {ext}")
    if nomes_colunas is not None:
        df.columns = nomes_colunas
    elif not tem_cabecalho:
        df.columns = [f"coluna_{i}" for i in range(df.shape[1])]
    return df


def dados_sinteticos(n=600, semente=SEMENTE):
    from sklearn.datasets import make_blobs
    X, _ = make_blobs(n_samples=n, centers=4, n_features=3, cluster_std=[1.0, 1.5, 0.8, 1.2], random_state=semente)
    return pd.DataFrame(X * [1, 10, 100], columns=["feature_1", "feature_2", "feature_3"])   # escalas diferentes


df = (dados_sinteticos() if CAMINHO_DADOS is None
      else carregar_tabela(CAMINHO_DADOS, SEPARADOR, TEM_CABECALHO, NOMES_COLUNAS))
features = COLUNAS_FEATURES or df.select_dtypes(include="number").columns.tolist()
dados = df[features].dropna()
print(f"{len(df) - len(dados)} linhas com faltantes removidas | formato usado: {dados.shape}")
escalador = StandardScaler() if PADRONIZAR else None
X = escalador.fit_transform(dados) if PADRONIZAR else dados.to_numpy(dtype=float)
display(dados.describe().T.round(3))

## 3. As duas etapas do K-Means
1. **Atribuição:** cada exemplo vai para o centroide mais próximo, $c^{(i)} := \arg\min_j \lVert\mathbf{x}^{(i)} - \boldsymbol\mu_j\rVert^2$ (índices de $0$ a $K-1$).
2. **Atualização:** cada centroide vira a média dos seus pontos, $\boldsymbol\mu_k = \frac{1}{|C_k|}\sum_{i\in C_k}\mathbf{x}^{(i)}$.

As duas etapas reduzem a distorção $J = \frac{1}{m}\sum_i \lVert\mathbf{x}^{(i)} - \boldsymbol\mu_{c^{(i)}}\rVert^2$ (a inércia do scikit-learn é a soma, sem dividir por $m$). O algoritmo sempre converge, mas para um mínimo **local**, que depende da inicialização.

In [ ]:
def find_closest_centroids(X, centroids):
    """Índice do centroide mais próximo de cada exemplo. X: (m, n), centroids: (K, n) → idx: (m,)."""
    d2 = ((X[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)     # (m, K) distâncias ao quadrado
    return np.argmin(d2, axis=1)


def find_closest_centroids_laco(X, centroids):
    """Versão em laço do original, mantida como referência."""
    idx = np.zeros(X.shape[0], dtype=int)
    for i in range(X.shape[0]):
        idx[i] = np.argmin([np.linalg.norm(X[i] - c) for c in centroids])
    return idx


def compute_centroids(X, idx, K, centroids_anteriores=None):
    """Média dos pontos de cada grupo. Grupo vazio: reposicionado no ponto mais distante do seu centroide."""
    n = X.shape[1]
    centroids = np.zeros((K, n))
    dist = None
    for k in range(K):
        pontos = X[idx == k]
        if len(pontos):
            centroids[k] = pontos.mean(axis=0)
        else:
            if dist is None:
                ref = centroids_anteriores if centroids_anteriores is not None else X[rng.choice(len(X), K, replace=False)]
                dist = ((X - ref[idx]) ** 2).sum(axis=1)
            j = int(np.argmax(dist))
            centroids[k] = X[j]
            dist[j] = -np.inf                         # o mesmo ponto não serve a dois grupos vazios
    return centroids


def compute_distortion(X, idx, centroids):
    return float(((X - centroids[idx]) ** 2).sum(axis=1).mean())


# Conferência com o caso do original: centroides iniciais fixos num conjunto pequeno.
Xc = np.array([[1.8, 4.6], [5.7, 4.8], [6.4, 3.3], [2.9, 4.6], [3.2, 4.9], [7.0, 3.0]])
c0 = np.array([[3, 3], [6, 2], [8, 5]], dtype=float)
assert (find_closest_centroids(Xc, c0) == find_closest_centroids_laco(Xc, c0)).all()
assert np.allclose(compute_centroids(Xc, np.array([0, 2, 1, 0, 0, 1]), 3)[0], Xc[[0, 3, 4]].mean(0))
assert not np.isnan(compute_centroids(Xc, np.zeros(6, int), 3, c0)).any(), "grupo vazio gerou nan"
print("atribuição (vetorizada = laço), atualização e grupo vazio conferidos")

## 4. Inicialização e execução
A inicialização aleatória do original escolhe *K* exemplos distintos por permutação. O **k-means++** escolhe o primeiro ao acaso e cada próximo com probabilidade proporcional ao quadrado da distância ao centroide mais próximo, o que espalha os pontos de partida e costuma convergir para soluções melhores.

In [ ]:
def kMeans_init_centroids(X, K):
    """Inicialização do original: K exemplos aleatórios, sem repetição."""
    return X[rng.permutation(X.shape[0])[:K]].copy()


def kmeans_pp_init(X, K):
    centroids = [X[rng.integers(len(X))]]
    for _ in range(1, K):
        d2 = ((X[:, None, :] - np.array(centroids)[None]) ** 2).sum(axis=2).min(axis=1)
        centroids.append(X[rng.choice(len(X), p=d2 / d2.sum())])
    return np.array(centroids)


def run_kMeans(X, initial_centroids, max_iters=MAX_ITERS, tol=TOLERANCIA, plot_progress=False, verbose=False):
    """Alterna atribuição e atualização até convergir. Devolve centroides, idx e o histórico da distorção."""
    centroids = initial_centroids.copy()
    K = len(centroids)
    trajetoria, historico = [centroids.copy()], []
    for i in range(max_iters):
        idx = find_closest_centroids(X, centroids)
        novos = compute_centroids(X, idx, K, centroids)
        historico.append(compute_distortion(X, idx, novos))
        movimento = np.abs(novos - centroids).max()
        centroids = novos
        trajetoria.append(centroids.copy())
        if verbose:
            print(f"Iteração {i}: distorção {historico[-1]:.5f}")
        if movimento < tol:
            break
    idx = find_closest_centroids(X, centroids)
    if plot_progress and X.shape[1] == 2:          # só desenha em 2-D, e só quando pedido
        plt.figure(figsize=(6, 4.5))
        plt.scatter(X[:, 0], X[:, 1], c=idx, s=10, alpha=0.6)
        traj = np.array(trajetoria)
        for k in range(K):
            plt.plot(traj[:, k, 0], traj[:, k, 1], "-x", c="k", lw=1)
        plt.title(f"Progresso do K-Means ({len(historico)} iterações)"); plt.show()
    return centroids, idx, historico


def kmeans(X, K, n_init=N_INICIALIZACOES, init=INICIALIZACAO):
    """Executa n_init vezes e fica com a solução de menor distorção."""
    melhor = None
    for _ in range(n_init):
        c0 = kmeans_pp_init(X, K) if init == "k-means++" else kMeans_init_centroids(X, K)
        c, idx, hist = run_kMeans(X, c0)
        if melhor is None or hist[-1] < melhor[2][-1]:
            melhor = (c, idx, hist)
    return melhor

In [ ]:
# Uma execução com progresso, como no original (o desenho aparece quando há 2 features).
c_demo, idx_demo, hist_demo = run_kMeans(X, kMeans_init_centroids(X, K), plot_progress=True, verbose=True)

## 5. Escolha de k: cotovelo e silhouette
O cartão recomenda olhar os dois juntos: a inércia cai sempre com *k*, então procure o "cotovelo" onde a queda desacelera; a silhouette (de −1 a 1) mede separação e coesão, maior é melhor. Decida também pela utilidade de negócio dos grupos.

In [ ]:
amostra = rng.choice(len(X), size=min(len(X), 3000), replace=False)   # silhouette é O(m²)
linhas = []
for k in FAIXA_K:
    c, idx, hist = kmeans(X, k, n_init=3)
    linhas.append({"k": k, "inércia": hist[-1] * len(X), "silhouette": silhouette_score(X[amostra], idx[amostra]),
                   "Davies-Bouldin": davies_bouldin_score(X, idx)})
escolha = pd.DataFrame(linhas).set_index("k")
display(escolha.round(3))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(escolha.index, escolha["inércia"], "o-"); ax[0].set(xlabel="k", ylabel="inércia", title="Cotovelo")
ax[1].plot(escolha.index, escolha["silhouette"], "o-"); ax[1].set(xlabel="k", ylabel="silhouette", title="Silhouette (maior é melhor)")
plt.tight_layout(); plt.show()
print("k de maior silhouette:", int(escolha["silhouette"].idxmax()), f"| K configurado: {K}")

## 6. Modelo final e perfil dos grupos

In [ ]:
centroids, idx, hist = kmeans(X, K)
print(f"distorção final: {hist[-1]:.5f} em {len(hist)} iterações")
centros_orig = escalador.inverse_transform(centroids) if PADRONIZAR else centroids
perfil = pd.DataFrame(centros_orig, columns=features)
perfil.insert(0, "tamanho", np.bincount(idx, minlength=K))
display(perfil.round(3))        # perfil médio de cada grupo, nas unidades originais

proj = X if X.shape[1] == 2 else PCA(n_components=2, random_state=SEMENTE).fit_transform(X)
cproj = centroids if X.shape[1] == 2 else PCA(n_components=2, random_state=SEMENTE).fit(X).transform(centroids)
plt.figure(figsize=(6, 4.5))
plt.scatter(proj[:, 0], proj[:, 1], c=idx, s=10, alpha=0.6)
plt.scatter(cproj[:, 0], cproj[:, 1], marker="x", s=150, linewidths=3, c="k")
plt.title("Grupos" + ("" if X.shape[1] == 2 else " (projeção PCA em 2-D)")); plt.show()

## 7. Validação com a referência (scikit-learn)
Mesmo *k* e mesma inicialização. Inércias próximas e índice de Rand ajustado perto de 1 (rótulos podem vir em outra ordem) confirmam a implementação.

In [ ]:
ref = KMeans(n_clusters=K, n_init=N_INICIALIZACOES, init="k-means++" if INICIALIZACAO == "k-means++" else "random",
             random_state=SEMENTE).fit(X)
inercia_zero = hist[-1] * len(X)
print(f"inércia · do zero: {inercia_zero:.3f} | scikit-learn: {ref.inertia_:.3f} "
      f"(diferença {100 * (inercia_zero - ref.inertia_) / ref.inertia_:+.2f}%)")
print(f"índice de Rand ajustado entre as duas partições: {adjusted_rand_score(ref.labels_, idx):.4f}")

## 8. Aplicação opcional: compressão de imagem
Cada pixel é um exemplo com 3 features (R, G, B). O K-Means escolhe as `K_IMAGEM` cores que melhor representam a imagem; cada pixel passa a guardar só o índice da sua cor. Sem padronização aqui: os canais já estão na mesma escala, de 0 a 1.

In [ ]:
if CAMINHO_IMAGEM is not None:
    original_img = plt.imread(CAMINHO_IMAGEM)
    if original_img.dtype == np.uint8:          # JPG e alguns PNG vêm de 0 a 255
        original_img = original_img / 255.0
    original_img = original_img[..., :3]          # descarta o canal alfa, se houver
    h, w, canais = original_img.shape
    X_img = original_img.reshape(h * w, canais)
    cores, idx_img, _ = kmeans(X_img, K_IMAGEM, n_init=1)
    X_recovered = cores[find_closest_centroids(X_img, cores)].reshape(original_img.shape)

    bits_orig = h * w * 24
    bits_comp = K_IMAGEM * 24 + h * w * int(np.ceil(np.log2(K_IMAGEM)))
    fig, ax = plt.subplots(1, 3, figsize=(14, 5), gridspec_kw={"width_ratios": [4, 4, 1]})
    ax[0].imshow(original_img); ax[0].set_title("Original"); ax[0].set_axis_off()
    ax[1].imshow(X_recovered); ax[1].set_title(f"Comprimida com {K_IMAGEM} cores"); ax[1].set_axis_off()
    ax[2].imshow(cores.reshape(-1, 1, 3), aspect="auto"); ax[2].set_title("Paleta"); ax[2].set_xticks([])
    plt.tight_layout(); plt.show()
    print(f"{bits_orig:,} bits → {bits_comp:,} bits (fator ≈ {bits_orig / bits_comp:.1f})")
else:
    print("CAMINHO_IMAGEM = None: seção pulada.")

## 9. Próximos passos (segundo o cartão do sistema)
- Pressupostos: grupos convexos, isotrópicos e de variância parecida; trate outliers antes e considere PCA em alta dimensão.
- **Prefira GMM** se as fronteiras forem suaves ou elípticas; **DBSCAN** se houver ruído e formas irregulares; **hierárquico** se quiser o dendrograma.
- Valide os grupos com a área de negócio: um *k* estatisticamente bom pode não ser útil.